# XRD Rietveld Plot Generator

Publication-quality Rietveld plots from the **CSV that the GSAS-II Rietveld
plot saves** - batch processing, built-in validation, cross-platform.

Not the file from *Export → Powder data as → histogram CSV file*: that one
has a quoted preamble and different column names, and is rejected.

Full documentation (input format, usage, configuration, privacy notes):
see [`README.md`](README.md).

## 1. Setup and core routines

Dependency check, then the parsing, data-preparation and plotting
functions. Plot appearance (2θ window, colours, line widths, fonts) is
controlled by the constants at the top of the second cell. Input format
and numerical-precision details are documented in the README.

In [ ]:
# Dependency bootstrap - installs only if missing (useful on Google Colab).
import importlib.util, subprocess, sys

for pkg in ("numpy", "pandas", "matplotlib", "ipywidgets"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])
print("Dependencies OK")

In [ ]:
"""The plotting engine lives in xrd_plotter.py; this cell loads it."""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xrd_plotter as xp

# Appearance is set by the constants in the module. Override them here, on
# the module itself, so every routine sees the change:
#   xp.PLOT_X_MIN, xp.PLOT_X_MAX = 13, 85   # fix the 2theta window
#   xp.WEIGHTED_RESIDUALS = False           # raw diff in the lower panel
#   xp.PHASE_COLORS = {"phase 1": "#1f77b4"}
print("Engine loaded:", Path(xp.__file__).name)


## 2. Validation (self-test on synthetic data)

Rebuilds synthetic GSAS-II-style exports and asserts bit-exact parsing in
both separator and decimal-mark variants, detection of the phase columns
among a full set of export columns, isolation of corrupt, ragged and
incomplete files, the order and colours of the phases, the metadata
binding into the legend, the 2theta window, the unweighted residual panel
and the function behind the interactive section. Raises `AssertionError`
on any failure, so the notebook doubles as an automated test
(`jupyter nbconvert --execute`). Only synthetic data is used.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    rng = np.random.default_rng(0)

    n = 500
    x = np.linspace(10.0, 90.0, n)
    calc = 1000.0 * np.exp(-((x - 30.0) ** 2) / 2.0) + 50.0
    obs = calc + rng.normal(0.0, 5.0, n)
    bkg = np.full(n, 50.0)
    ds = (obs - calc) / 5.0

    df = pd.DataFrame({"2theta": x, "Obs": obs, "Calc": calc, "Bkg": bkg,
                       "diff/sigma": ds})
    df["Phase 1 hkl"] = pd.Series([30.0, 35.0, 50.0, 60.0])
    df["Phase 2 hkl"] = pd.Series([32.0, 38.0, 52.0])

    # Variant A: semicolon separator + decimal commas (locale export).
    f_a = tmp / "sample_A.csv"
    df.to_csv(f_a, sep=";", index=False, decimal=",")
    # Variant B: comma separator, decimal points, 'x, deg' header.
    f_b = tmp / "sample_B.csv"
    df.rename(columns={"2theta": "x, deg"}).to_csv(f_b, index=False)

    for f in (f_a, f_b):
        data, phase_cols, error = xp.read_gsas2_csv(f)
        assert error is None, f"{f.name}: {error}"
        # 1. bit-exact round trip of the parsed doubles
        assert np.array_equal(data["x"], x), "2theta round trip not exact"
        assert np.array_equal(data["obs"], obs), "Obs round trip not exact"
        assert np.array_equal(data["resid"], ds), "diff/sigma not exact"
        # 2. structure detection
        assert sorted(phase_cols) == ["Phase 1 hkl", "Phase 2 hkl"]
        print(f"{f.name}: bit-exact parse, 2 phases - OK")

    # Degraded inputs must be isolated, not fatal.
    (tmp / "corrupt.csv").write_bytes(b"\x00\x01\x02 not a csv \xff")
    _, _, err = xp.read_gsas2_csv(tmp / "corrupt.csv")
    assert err is not None
    print(f"corrupt.csv isolated: {err.splitlines()[0][:60]}... - OK")

    (tmp / "no_theta.csv").write_text("A;B\n1;2\n")
    _, _, err = xp.read_gsas2_csv(tmp / "no_theta.csv")
    assert err == "2theta column not found"
    print("no_theta.csv isolated: 2theta column not found - OK")

    f_min = tmp / "only_obs.csv"
    df[["2theta", "Obs"]].to_csv(f_min, sep=";", index=False)
    _, _, err = xp.read_gsas2_csv(f_min)
    assert err == "residual column 'diff/sigma' not found", err
    print("only_obs.csv isolated: no residual column - OK")

    f_part = tmp / "no_calc.csv"
    df[["2theta", "Obs", "diff/sigma"]].to_csv(f_part, sep=";", index=False)
    data_p, _, err = xp.read_gsas2_csv(f_part)
    assert err is None and np.all(data_p["calc"] == 0.0)
    print("no_calc.csv: calc and bkg padded with zeros, still drawn - OK")

    f_nan = tmp / "obs_all_text.csv"
    df.assign(Obs="n/a").to_csv(f_nan, sep=";", index=False)
    _, _, err = xp.read_gsas2_csv(f_nan)
    assert err == "no valid data in the obs column", err
    print("obs_all_text.csv isolated: nothing left to draw - OK")

    # 3. a file the C parser cannot tokenise: one row carries two fields
    # too many. The fallback drops that row and keeps the rest exact.
    lines = f_a.read_text().splitlines()
    lines[5] += ";99;99"
    f_ragged = tmp / "ragged.csv"
    f_ragged.write_text("\n".join(lines) + "\n")
    data_r, phase_r, err = xp.read_gsas2_csv(f_ragged)
    assert err is None, f"ragged.csv: {err}"
    assert np.array_equal(data_r["obs"], np.delete(obs, 4)), "fallback not exact"
    assert sorted(phase_r) == ["Phase 1 hkl", "Phase 2 hkl"], phase_r
    print("ragged.csv: one row skipped, the rest bit-exact - OK")

    # 4. a complete export, with the header the README documents and in the
    # column order GSAS-II writes: the sparse bookkeeping columns
    # (tick-pos, Axis-limits) must not be mistaken for phases, a repeated
    # header must not either, and a column too full to be a reflection list
    # is reported rather than dropped in silence.
    full = pd.DataFrame({"used": np.ones(n), "x, 2theta (deg)": x, "obs": obs,
                         "calc": calc, "bkg": bkg, "diff": obs - calc})
    full["Alpha hkl"] = pd.Series([30.0, 35.0, 50.0, 60.0])
    full["Beta hkl"] = pd.Series([32.0, 38.0, 52.0])
    full["tick-pos"] = pd.Series([-0.5])
    full["diff/sigma"] = ds
    full["Axis-limits"] = pd.Series([10.0, 90.0])
    f_full = tmp / "full_export.csv"
    full.to_csv(f_full, sep=";", index=False)
    data_full, phase_cols_full, err = xp.read_gsas2_csv(f_full)
    assert err is None, f"full_export.csv: {err}"
    assert sorted(phase_cols_full) == ["Alpha hkl", "Beta hkl"], phase_cols_full
    print(f"full_export.csv: {len(full.columns)} columns, 2 phases - OK")

    f_dup = tmp / "repeated_header.csv"
    f_dup.write_text("2theta;Obs;diff/sigma;tick-pos;tick-pos\n"
                     + "".join(f"{v};1;0;;\n" for v in (10.0, 20.0, 30.0))
                     .replace("10.0;1;0;;", "10.0;1;0;-0.5;-0.5"))
    _, phase_dup, err = xp.read_gsas2_csv(f_dup)
    assert err is None and phase_dup == [], phase_dup
    print("repeated_header.csv: 'tick-pos.1' is not a phase - OK")

    f_dense = tmp / "dense_column.csv"
    dense = df.copy()
    dense["Wide hkl"] = pd.Series(np.linspace(10.0, 90.0, n - 100))
    dense.to_csv(f_dense, sep=";", index=False)
    _, phase_dense, err = xp.read_gsas2_csv(f_dense)
    assert err is None and "Wide hkl" not in phase_dense, phase_dense
    print("dense_column.csv: too full for a reflection list, reported - OK")

    # 5. metadata wiring into the legend (synthetic metadata only).
    meta = tmp / "Samples_metadata.csv"
    meta.write_text("filename;formula;phase_1_pct;phase_2_pct\n"
                    "sample_A.csv;Sample A (synthetic);60;40\n")
    meta_df = xp.load_metadata(meta)
    name, pct, colors, window = xp.sample_info(meta_df, "sample_A.csv", "sample_A")
    assert name == "Sample A (synthetic)"
    assert pct == {"phase 1": 60.0, "phase 2": 40.0}, pct
    assert colors == {} and window == (None, None)

    data, phase_cols, _ = xp.read_gsas2_csv(f_a)
    theta, o, c, b, r, phases = xp.prepare_data(data, phase_cols, use_sqrt=True)
    fig = xp.create_plot(theta, o, c, b, r, phases, name, pct, use_sqrt=True)
    labels = [t.get_text() for t in fig.axes[0].get_legend().get_texts()]
    assert "Phase 1 (60%)" in labels and "Phase 2 (40%)" in labels
    assert name in labels
    print("legend labels:", labels)
    plt.show()
    plt.close(fig)

    # 6. phase names, order and colours: alphabetical by legend name, one
    # tick row per phase, the cycle handed out in that order. Two columns
    # of one phase share an entry, a colour and a row.
    assert xp.phase_label("Phase 1 hkl") == "Phase 1"
    assert xp.phase_label("alpha") == "Alpha"
    phases3 = dict(phases, **{"Phase 3 hkl": np.array([40.0, 45.0])})
    fig3 = xp.create_plot(theta, o, c, b, r, phases3, name, pct, use_sqrt=True)
    ax3 = fig3.axes[0]
    assert [t.get_text() for t in ax3.get_legend().get_texts()][3:] == [
        "Phase 1 (60%)", "Phase 2 (40%)", "Phase 3"]
    assert [h.get_color() for h in ax3.get_legend().legend_handles[3:]] == list(
        xp.PHASE_COLOR_CYCLE[:3])
    assert len([c for c in ax3.collections if hasattr(c, "get_segments")]) == 3
    plt.close(fig3)

    twin = dict(phases, **{"Phase 1": phases["Phase 1 hkl"]})
    fig_t = xp.create_plot(theta, o, c, b, r, twin, name, pct, use_sqrt=True)
    ax_t = fig_t.axes[0]
    assert [t.get_text() for t in ax_t.get_legend().get_texts()][3:] == [
        "Phase 1 (60%)", "Phase 2 (40%)"]
    assert [h.get_color() for h in ax_t.get_legend().legend_handles[3:]] == list(
        xp.PHASE_COLOR_CYCLE[:2]), "a repeated label must not eat a colour"
    print("phase order, colour cycle and repeated labels - OK")
    plt.close(fig_t)

    # 7. a colour from the metadata follows its phase, whether or not the
    # other phases are in that sample, and the longest matching key wins.
    meta_col = tmp / "metadata_colour.csv"
    meta_col.write_text("filename;phase_2_color;phase_1_color\n"
                        "sample_A.csv;#123456;not a colour\n")
    _, _, colors_c, _ = xp.sample_info(xp.load_metadata(meta_col), "sample_A.csv",
                                    "sample_A")
    assert colors_c == {"phase 2": "#123456"}, colors_c  # bad cell dropped
    solo = {"Phase 2 hkl": phases["Phase 2 hkl"]}
    fig1 = xp.create_plot(theta, o, c, b, r, solo, name, pct, use_sqrt=True,
                       colors=colors_c)
    assert fig1.axes[0].get_legend().legend_handles[3].get_color() == "#123456"
    fig2 = xp.create_plot(theta, o, c, b, r, phases, name, pct, use_sqrt=True,
                       colors=colors_c)
    assert [h.get_color() for h in fig2.axes[0].get_legend().legend_handles[3:]
            ] == [xp.PHASE_COLOR_CYCLE[0], "#123456"]
    assert xp.longest_match({"phase": "#000000", "phase 1": "#ffffff"},
                         "Phase 1") == "#ffffff"
    assert xp.metadata_keys(["_pct", "phase_1_pct"], "_pct") == {
        "phase 1": "phase_1_pct"}, "a suffix-only column matches every phase"
    print("metadata colour: pinned per phase, longest key wins - OK")
    plt.close(fig1)
    plt.close(fig2)

    # 8. the two dictionaries in the routines cell: a legend name, and a
    # colour keyed by the name that is printed.
    xp.PHASE_LABELS = {"phase 1": "Alpha"}
    xp.PHASE_COLORS = {"alpha": "#654321"}
    assert xp.phase_label("Phase 1 hkl") == "Alpha"
    fig_d = xp.create_plot(theta, o, c, b, r, phases, name, pct, use_sqrt=True)
    handles_d = fig_d.axes[0].get_legend().legend_handles[3:]
    texts_d = [t.get_text() for t in fig_d.axes[0].get_legend().get_texts()][3:]
    assert texts_d == ["Alpha", "Phase 2 (40%)"], texts_d
    assert [h.get_color() for h in handles_d] == ["#654321",
                                                  xp.PHASE_COLOR_CYCLE[0]]
    xp.PHASE_LABELS, xp.PHASE_COLORS = {}, {}
    print("xp.PHASE_LABELS and xp.PHASE_COLORS: renamed phase keeps its colour - OK")
    plt.close(fig_d)

    # 9. metadata quirks: a decimal comma, an empty formula cell that must
    # not reach the legend as 'nan', a percentage column matching no phase,
    # and two columns matching one phase.
    meta_v = tmp / "metadata_variants.csv"
    meta_v.write_text("filename;formula;phase_1_pct;phase_9_pct\n"
                      "sample_A.csv;;60,5;10\n")
    name_v, pct_v, _, _ = xp.sample_info(xp.load_metadata(meta_v), "sample_A.csv",
                                      "sample_A")
    assert name_v == "sample_A", name_v
    assert pct_v == {"phase 1": 60.5, "phase 9": 10.0}, pct_v
    fig_v = xp.create_plot(theta, o, c, b, r, phases, name_v, pct_v, use_sqrt=True)
    lab_v = [t.get_text() for t in fig_v.axes[0].get_legend().get_texts()]
    assert lab_v[0] == "sample_A" and "Phase 1 (60.5%)" in lab_v, lab_v
    assert fig_v.axes[1].get_ylabel() == r"diff/$\sigma$"
    assert xp.phase_fraction({"phase": 40.0, "phase 1": 45.0}, "Phase 1") == 0.0
    print("metadata variants: comma, empty formula, orphan, collision - OK")
    plt.close(fig_v)

    # 10. the 2theta window: the measured range by default, the constants
    # over it, the metadata pair over both, and the intensity axis scaled
    # on what the window shows.
    assert xp.plot_window(theta) == (10.0, 90.0), xp.plot_window(theta)
    xp.PLOT_X_MIN, xp.PLOT_X_MAX = 13, 85
    assert xp.plot_window(theta) == (13.0, 85.0), xp.plot_window(theta)
    assert xp.plot_window(theta, 20.0, 60.0) == (20.0, 60.0)
    xp.PLOT_X_MIN, xp.PLOT_X_MAX = None, None
    meta_w = tmp / "metadata_window.csv"
    meta_w.write_text("filename;x_min;x_max\nsample_A.csv;20;60,5\n")
    _, _, _, window_w = xp.sample_info(xp.load_metadata(meta_w), "sample_A.csv",
                                    "sample_A")
    assert window_w == (20.0, 60.5), window_w
    fig_w = xp.create_plot(theta, o, c, b, r, phases, name, pct, use_sqrt=True,
                        xlim=window_w)
    assert fig_w.axes[1].get_xlim() == (20.0, 60.5)
    full_top = xp.create_plot(theta, o, c, b, r, phases, name, pct,
                           use_sqrt=True).axes[0].get_ylim()[1]
    assert fig_w.axes[0].get_ylim()[1] < full_top, "y not rescaled to the window"
    print("plot window: data range, constants, metadata, y rescaled - OK")
    plt.close("all")

    # 11. the unweighted panel reads the raw diff, labels it in counts, and
    # refuses a file that does not carry the column it was asked for.
    data_u, phase_u, err = xp.read_gsas2_csv(f_full, weighted=False)
    assert err is None, err
    assert np.array_equal(data_u["resid"], obs - calc), "diff not exact"
    fig_u = xp.create_plot(*xp.prepare_data(data_u, phase_u, use_sqrt=True),
                        name, pct, use_sqrt=True, weighted=False)
    assert fig_u.axes[1].get_ylabel() == r"diff / (a.u.)"
    plt.close(fig_u)
    _, _, err = xp.read_gsas2_csv(f_a, weighted=False)
    assert err == "residual column 'diff' not found", err
    assert xp.WEIGHTED_RESIDUALS is True, "the constant must not be mutated"
    print("unweighted residuals: raw diff, own label, missing column - OK")

    # 12. the function behind the interactive panel: limits applied, the
    # metadata line handed back, the setting left alone.
    fig_i, line_i = xp.replot_file(f_a, meta, x_min=20, x_max=60.5, y_max=40,
                                use_sqrt=True, weighted=True)
    assert fig_i.axes[1].get_xlim() == (20.0, 60.5)
    assert fig_i.axes[0].get_ylim()[1] == 40.0
    assert line_i.splitlines()[1] == "sample_A.csv;Sample A (synthetic);20;60.5"
    plt.close(fig_i)
    fig_j, _ = xp.replot_file(f_full, meta, weighted=False)
    assert fig_j.axes[1].get_ylabel() == r"diff / (a.u.)"
    assert xp.WEIGHTED_RESIDUALS is True, "the constant must not be mutated"
    plt.close(fig_j)
    try:
        xp.replot_file(f_min, meta)
    except ValueError as e:
        assert "residual column" in str(e), e
    else:
        raise AssertionError("a file without residuals must raise")
    print("xp.replot_file: limits applied, metadata line, no side effect - OK")

    # 13. one unusable file must not end the batch: the files after it are
    # still drawn, and it is reported with a reason.
    batch_in, batch_out = tmp / "batch_in", tmp / "batch_out"
    batch_in.mkdir()
    df.to_csv(batch_in / "a_good.csv", sep=";", index=False)
    df.assign(Obs="n/a").to_csv(batch_in / "b_broken.csv", sep=";", index=False)
    df.to_csv(batch_in / "c_good.csv", sep=";", index=False)
    outcome = xp.process_folder(batch_in, meta, batch_out, show=False)
    assert [status for _, status, _ in outcome] == ["ok", "error", "ok"], outcome
    assert len(list(batch_out.glob("*.png"))) == 2
    print("batch: the broken file is reported, the others are drawn - OK")

print("\nALL VALIDATION CHECKS PASSED")

## 3. Plot your own exports

Copy your CSV exports into `data/`, optionally place `Samples_metadata.csv`
next to the notebook, adjust the parameters below and run. Figures are
shown inline and saved to `output/` as PDF and 600 dpi PNG. Step-by-step
instructions (local and Google Colab) are in the README.

> **Keep your data private:** `data/`, `output/` and `Samples_metadata.csv`
> are listed in `.gitignore` and must never be committed or uploaded.

In [ ]:
DATA_FOLDER = "data"                       # your GSAS-II CSV exports
METADATA_FILE = "Samples_metadata.csv"     # optional, PRIVATE - never commit
OUTPUT_FOLDER = "output"                   # created automatically
USE_SQRT = True                            # False -> linear intensity axis

results = xp.process_folder(DATA_FOLDER, METADATA_FILE, OUTPUT_FOLDER,
                         use_sqrt=USE_SQRT)

## 4. Try a different window on one file

Pick a file, type the limits, press **Apply**. Nothing is saved: this is
the place to find the window you want before running the batch again.
An empty box leaves that end of the axis to the setting behind it, the
`PLOT_X_MIN` and `PLOT_X_MAX` constants for 2theta and the data itself
for the intensity.

The panel prints a metadata line under the figure. Paste it into
`Samples_metadata.csv` and section 3 will draw that sample this way every
time.

This section needs `ipywidgets`, which the first cell installs along with
the other dependencies. Without it the section prints how to install it
and the rest of the notebook is unaffected.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except ImportError:
    widgets = None
    print("ipywidgets is not installed: run 'pip install ipywidgets', "
          "then re-run this cell.")

files = sorted(f for f in Path(DATA_FOLDER).glob("*.csv")
               if f.name != Path(METADATA_FILE).name)

if widgets is None or not files:
    if widgets is not None:
        print(f"No CSV files in '{DATA_FOLDER}': nothing to replot.")
else:
    picker = widgets.Dropdown(options=[(f.name, str(f)) for f in files],
                              description="File:")
    boxes = {k: widgets.Text(description=d, placeholder="auto",
                             layout=widgets.Layout(width="180px"))
             for k, d in (("x_min", "2theta min"), ("x_max", "2theta max"),
                          ("y_min", "y min"), ("y_max", "y max"))}
    sqrt_box = widgets.Checkbox(value=USE_SQRT, description="sqrt intensity")
    weighted_box = widgets.Checkbox(value=xp.WEIGHTED_RESIDUALS,
                                    description="diff/sigma")
    apply_button = widgets.Button(description="Apply", button_style="primary")
    out = widgets.Output()

    def on_apply(_):
        """Redraw the picked file with whatever the boxes currently hold."""
        with out:
            clear_output(wait=True)
            limits = {k: xp.to_number(b.value) if b.value.strip() else None
                      for k, b in boxes.items()}
            try:
                fig, line = xp.replot_file(picker.value, METADATA_FILE,
                                        use_sqrt=sqrt_box.value,
                                        weighted=weighted_box.value, **limits)
            except ValueError as e:
                print(f"Cannot draw this file: {e}")
                return
            plt.show()
            plt.close(fig)
            print("Metadata line for this window:\n" + line)

    apply_button.on_click(on_apply)
    display(widgets.VBox([
        picker,
        widgets.HBox([boxes["x_min"], boxes["x_max"]]),
        widgets.HBox([boxes["y_min"], boxes["y_max"]]),
        widgets.HBox([sqrt_box, weighted_box, apply_button]),
        out,
    ]))